# From Pauli matrices to any SU(N): gauge representations, derived and checked

*For someone new to particle physics — you need only linear algebra and a little quantum
mechanics.*

Quarks come in **three colours**. The weak force acts on **doublets**. Grand unified
theories pack a whole matter generation into a $\bar{\mathbf 5}$ **and a** $\mathbf{10}$ of
SU(5). Every one of these statements is about a **representation of a Lie group** — the
mathematical language of gauge symmetry. `DiscreteGroups_Tutorial` does the same job for
*finite* flavour groups; this is the **continuous** counterpart.

The pedagogy is **compute step by step, then develop judgment**. Nothing below is typed in
from the answer: the structure constants are *derived* from the matrices, the adjoint is
*built* from them, the number of singlets in a product is *predicted* and then *checked*
against explicit construction — and only then compared with what `feynlag` builds.

Each section runs four moves — **set up** (look at the raw object), **collect** (let the
structure emerge), **recognise** (name what appeared and why it had to), **check** (from
group theory, not by re-running the same algebra) — and a blockquote asks you to commit to
an expectation *before* the cell that settles it.

**The one question this notebook is really about.** You have fields in some
representations. Which combinations of them can appear in a gauge-invariant Lagrangian — a
mass term, a Yukawa, a meson, a baryon? You can answer that **before writing a single
term**, and every tool below is a step towards doing so.

**Roadmap**

1. Generators are the group, infinitesimally
2. The Lie algebra
3. One algebra, many sizes
4. Labelling representations
5. Inside a representation: weights and ladders
6. What survives a change of basis
7. Conjugates, and three kinds of reality
8. Counting singlets before building them
9. Anomalies
10. Into a model
11. Capstone: the $\mathbf 6$ of SU(3) by hand
12. Recap
13. References

In [ ]:
import itertools

import sympy as sp
from sympy.physics.quantum import TensorProduct
sp.init_printing(use_latex="mathjax")   # text/latex only -- no PNG mimetype

from feynlag import (SU2, SU3, SUN, Scalar, WeylFermion, Model,
                     ExternalParameter, Lagrangian, Dmu,
                     structure_constants, check_anomaly_free)
from feynlag.groups import sun


def ok(msg):
    print("✓", msg)


def trace(msg, obj=None):
    # One line of commentary, optionally followed by a sympy object.  Every
    # helper below takes `verbose=False` and routes its narration through
    # here, so the tracing style is uniform and switched off by default --
    # the cells that assert stay quiet, and the "peek" cells do the talking.
    print(msg)
    if obj is not None:
        display(obj)

---
## 1. Generators are the group, infinitesimally

A **symmetry** acts on a *multiplet* of fields — a column of $n$ objects that the
transformation mixes among themselves, i.e. multiplication by an $n\times n$ matrix $U$.
For a **continuous** symmetry you can turn the transformation on gently. Close to "doing
nothing",

$$U(\alpha) \;=\; e^{\,i\alpha^a T^a} \;\approx\; \mathbb 1 + i\,\alpha^a T^a + \dots$$

The matrices $T^a$ at first order are the **generators**. A finite group was its
generators *multiplied*; a Lie group is its generators *exponentiated*.

The smallest non-trivial example is **SU(2)**, acting on a doublet such as $(\nu_e, e)_L$.
In that 2-dimensional (*fundamental*) representation the generators are the Pauli matrices
over two. Write them down rather than importing them, then check the library agrees.

> **Before running the next cells.** $U$ must be **unitary** (it preserves probability)
> and have **determinant one** (the "S" in SU). Using $U\approx\mathbb1+i\alpha T$, work out
> what each condition demands of $T$ to first order in $\alpha$. You should land on two
> properties — and you will see both asserted below.

In [ ]:
# ---- MOVE 1: set up.  The Pauli matrices over two, by hand. ---------------
sigma = [sp.Matrix([[0, 1], [1, 0]]),
         sp.Matrix([[0, -sp.I], [sp.I, 0]]),
         sp.Matrix([[1, 0], [0, -1]])]
T2 = [s / 2 for s in sigma]

SU2L = SU2("SU2L")
assert SU2L.generators(2) == T2
display(*T2)
ok("feynlag's SU(2) doublet generators are exactly sigma_a / 2")

⎡ 0   1/2⎤
⎢        ⎥
⎣1/2   0 ⎦

⎡   -ⅈ ⎤
⎢0  ───⎥
⎢    2 ⎥
⎢      ⎥
⎢ⅈ     ⎥
⎢─   0 ⎥
⎣2     ⎦

⎡1/2   0  ⎤
⎢         ⎥
⎣ 0   -1/2⎦

✓ feynlag's SU(2) doublet generators are exactly sigma_a / 2


In [ ]:
# ---- MOVE 2 + 4: the two conditions from the box. -------------------------
for a, Ta in enumerate(T2):
    assert Ta == Ta.H, a                 # unitary U      <=>  hermitian T
    assert sp.trace(Ta) == 0, a          # det U = 1      <=>  traceless T
ok("every generator is hermitian (U unitary) and traceless (det U = 1)")

✓ every generator is hermitian (U unitary) and traceless (det U = 1)


The box's derivation: $U^\dagger U=\mathbb 1$ at first order gives $-i\alpha T^\dagger + i\alpha
T=0$, so $T=T^\dagger$; and $\det e^{i\alpha T}=e^{i\alpha\,\mathrm{Tr}\,T}=1$ forces
$\mathrm{Tr}\,T=0$. The first-order conditions are not an approximation to be improved
later — exponentiating makes them exact, which the next cell shows rather than asserts by
eye.

In [ ]:
# ---- peek: exponentiate one generator, and check it is in SU(2) exactly ----
alpha = sp.Symbol("alpha", real=True)
U_alpha = sp.simplify((sp.I * alpha * T2[0]).exp())
trace("exp(i alpha T^1) =", U_alpha)
assert sp.simplify(U_alpha * U_alpha.H) == sp.eye(2)
assert sp.simplify(U_alpha.det()) == 1
ok("exp(i alpha T^1) is unitary with determinant 1 for EVERY alpha, not just small ones")

exp(i alpha T^1) =


⎡    ⎛α⎞        ⎛α⎞⎤
⎢ cos⎜─⎟   ⅈ⋅sin⎜─⎟⎥
⎢    ⎝2⎠        ⎝2⎠⎥
⎢                  ⎥
⎢     ⎛α⎞      ⎛α⎞ ⎥
⎢ⅈ⋅sin⎜─⎟   cos⎜─⎟ ⎥
⎣     ⎝2⎠      ⎝2⎠ ⎦

✓ exp(i alpha T^1) is unitary with determinant 1 for EVERY alpha, not just small ones


---
## 2. The Lie algebra: how generators multiply

Generators do not commute — that is what makes the group *non-abelian* and the physics
rich. How they fail to commute is the single most important piece of data about the group:

$$[T^a, T^b] \;=\; i\, f^{abc}\, T^c .$$

The numbers $f^{abc}$ are the **structure constants**. Rather than look them up, extract
them. With the standard normalisation $\mathrm{Tr}(T^aT^b)=\tfrac12\delta^{ab}$, multiply
the commutator by $T^c$ and take the trace:

$$f^{abc} \;=\; -2i\,\mathrm{Tr}\big([T^a,T^b]\,T^c\big).$$

> **Before running the next cell.** For SU(2), compute $[T^1,T^2]$ by hand from the
> matrices above. Which generator comes out, with what coefficient? Then guess the whole
> pattern: which $f^{abc}$ are non-zero, and what is their sign under swapping two indices?

In [ ]:
# ---- MOVE 1 + 2: derive f^{abc} from the matrices themselves. -------------
def derive_f(T, verbose=False):
    # Non-zero f^{abc} = -2i Tr([T^a, T^b] T^c), assuming Tr(T^a T^b) = delta/2.
    n, out = len(T), {}
    for a, b in itertools.product(range(n), repeat=2):
        comm = T[a] * T[b] - T[b] * T[a]
        for c in range(n):
            v = sp.nsimplify(sp.expand(-2 * sp.I * sp.trace(comm * T[c])))
            if v != 0:
                out[(a, b, c)] = v
                if verbose and a < b < c:
                    trace(f"  f^({a + 1}{b + 1}{c + 1}) = {v}")
    return out


f2 = derive_f(T2)
assert f2 == structure_constants(SU2L)
assert all(f2.get(p, 0) == sp.LeviCivita(*p) for p in itertools.product(range(3), repeat=3))
ok("derived from the matrices: f^{abc} = epsilon^{abc} for SU(2), and it matches "
   "structure_constants(SU2L)")

✓ derived from the matrices: f^{abc} = epsilon^{abc} for SU(2), and it matches structure_constants(SU2L)


In [ ]:
# ---- peek: the same extraction for SU(3), narrating the independent entries -
SU3c = SU3("SU3c")
T3 = SU3c.generators(3)
f3 = derive_f(T3, verbose=True)

  f^(123) = 1
  f^(147) = 1/2
  f^(156) = -1/2
  f^(246) = 1/2
  f^(257) = 1/2
  f^(345) = 1/2
  f^(367) = -1/2


  f^(458) = sqrt(3)/2
  f^(678) = sqrt(3)/2


In [ ]:
# ---- MOVE 4: check against the library, and against the defining relation --
assert f3 == {k: sp.nsimplify(v) for k, v in structure_constants(SU3c).items()}
assert sp.Matrix(8, 8, lambda a, b: sp.trace(T3[a] * T3[b])) == sp.eye(8) / 2
# f is totally antisymmetric: swapping any two indices flips the sign
for (a, b, c), v in f3.items():
    assert f3.get((b, a, c)) == -v and f3.get((a, c, b)) == -v
ok("SU(3): 54 non-zero f^{abc}, totally antisymmetric, identical to structure_constants")

✓ SU(3): 54 non-zero f^{abc}, totally antisymmetric, identical to structure_constants


**Recognise.** Nine independent values ($a<b<c$) generate all 54 entries by antisymmetry,
and only three distinct magnitudes appear: $1$, $\tfrac12$ and $\tfrac{\sqrt3}{2}$.

**The judgment to keep:** the normalisation $\mathrm{Tr}(T^aT^b)=\tfrac12\delta^{ab}$ is a
*choice*, made once, for the fundamental. Rescale every $T$ by $\lambda$ and every $f$
rescales by $\lambda$ too. Once $f$ is fixed, though, nothing else is free: §3 shows every
other representation is forced to use the *same* $f$, so its normalisation is no longer a
choice at all.

---
## 3. One algebra, many sizes

The *same* abstract symmetry can act on multiplets of **different sizes**. Each such
realisation is a **representation**: a set of matrices obeying the **same Lie algebra**,
i.e. with the **same** $f^{abc}$. Three exist for every SU($N$):

- the **singlet** — $1\times1$, $T^a=0$: a field *neutral* under the force;
- the **fundamental** — $N\times N$; quarks live in the $\mathbf 3$ of colour SU(3);
- the **adjoint** — one dimension per generator, built out of $f$ itself:
  $(T^a_{\rm adj})_{bc} = -i f^{abc}$. The gauge bosons (gluons) live here.

> **Before running the next cell.** The adjoint of SU(3) is built from the 54 numbers you
> just derived and nothing else. Why should eight $8\times8$ matrices made from $f$ obey
> the algebra whose structure constants *are* $f$? (Hint: the Jacobi identity
> $[T^a,[T^b,T^c]]+\text{cyclic}=0$ is a statement about $f$ alone.)

In [ ]:
# ---- MOVE 1: build the adjoint by hand from f, then compare. --------------
adj_hand = [sp.Matrix(8, 8, lambda b, c: -sp.I * f3.get((a, b, c), 0))
            for a in range(8)]
assert adj_hand == [sp.nsimplify(M) for M in SU3c.generators(8)]
ok("(T^a)_bc = -i f^{abc}, built from the derived f, is exactly feynlag's SU(3) octet")

✓ (T^a)_bc = -i f^{abc}, built from the derived f, is exactly feynlag's SU(3) octet


**What an adjoint generator looks like.** Open one. Every entry is $-i$ times a structure
constant, so the matrix is purely imaginary and antisymmetric — which is how it manages to
be hermitian.

In [ ]:
adj_hand[0]

⎡0  0  0   0   0   0   0   0⎤
⎢                           ⎥
⎢0  0  -ⅈ  0   0   0   0   0⎥
⎢                           ⎥
⎢0  ⅈ  0   0   0   0   0   0⎥
⎢                           ⎥
⎢                     -ⅈ    ⎥
⎢0  0  0   0   0   0  ───  0⎥
⎢                      2    ⎥
⎢                           ⎥
⎢                  ⅈ        ⎥
⎢0  0  0   0   0   ─   0   0⎥
⎢                  2        ⎥
⎢                           ⎥
⎢             -ⅈ            ⎥
⎢0  0  0   0  ───  0   0   0⎥
⎢              2            ⎥
⎢                           ⎥
⎢          ⅈ                ⎥
⎢0  0  0   ─   0   0   0   0⎥
⎢          2                ⎥
⎢                           ⎥
⎣0  0  0   0   0   0   0   0⎦

Row $b$, column $c$ holds $-if^{1bc}$: the non-zero entries sit exactly where the first
generator's commutators land. Nothing about "gluons" was put in; the octet is the algebra
acting on itself.

In [ ]:
# ---- MOVE 2 + 4: the same algebra at every size. ---------------------------
def closes(T, f, verbose=False):
    # [T^a, T^b] = i f^{abc} T^c for every pair, with the SHARED f.
    n, dim = len(T), T[0].shape[0]
    for a, b in itertools.combinations(range(n), 2):
        lhs = T[a] * T[b] - T[b] * T[a]
        rhs = sum((sp.I * f.get((a, b, c), 0) * T[c] for c in range(n)),
                  sp.zeros(dim, dim))
        if sp.expand(lhs - rhs) != sp.zeros(dim, dim):
            if verbose:
                trace(f"  [T^{a + 1}, T^{b + 1}] fails")
            return False
    if verbose:
        trace(f"  all {n * (n - 1) // 2} commutators close on dimension {dim}")
    return True


for label, T in (("singlet 1", SU3c.generators(1)), ("fundamental 3", T3),
                 ("adjoint 8", adj_hand)):
    assert closes(T, f3), label
    print(f"  {label:<14} {T[0].shape}  closes with the shared f")
ok("1x1, 3x3 and 8x8 matrices -- three sizes, one algebra")

  singlet 1      (1, 1)  closes with the shared f
  fundamental 3  (3, 3)  closes with the shared f


  adjoint 8      (8, 8)  closes with the shared f
✓ 1x1, 3x3 and 8x8 matrices -- three sizes, one algebra


---
## 4. Labelling representations: dimension and Dynkin labels

So far the labels were **dimensions**: `1`, `3`, `8`. That is convenient but incomplete —
a group can have *several* representations of the same dimension. `feynlag` builds any
SU($N$) with `SUN(N, name)` and accepts three kinds of label:

- an **integer dimension** (`1`, `N`, `N`$^2-1$, or any unambiguous dimension);
- a **Dynkin label** — a tuple $(a_1,\dots,a_{N-1})$ of non-negative integers naming
  exactly one irrep;
- a **conjugate** label (§7).

The dimension of the irrep with a given Dynkin label follows from the **Weyl dimension
formula** [Weyl25] — pure combinatorics, no matrices built.

> **Before running the next cell.** For SU(3), the Dynkin label $(p,q)$ has dimension
> $\tfrac12(p+1)(q+1)(p+q+2)$. Tabulate $p+q\le 4$ by hand. Which dimensions repeat? Is
> there *any* integer that is the dimension of two genuinely different irreps that are
> **not** each other's conjugate?

In [ ]:
# ---- MOVE 1 + 2: tabulate, and group by dimension. ------------------------
by_dim = {}
for p, q in itertools.product(range(5), repeat=2):
    d = sun.weyl_dim(3, (p, q))
    assert d == (p + 1) * (q + 1) * (p + q + 2) // 2
    by_dim.setdefault(d, []).append((p, q))

for d in sorted(by_dim)[:9]:
    print(f"  dim {d:>3} : {by_dim[d]}")
shared = {d: labs for d, labs in by_dim.items()
          if len({tuple(sorted(l)) for l in labs}) > 1}
print("\n  dimensions shared by NON-conjugate irreps:", shared)
assert shared == {15: [(0, 4), (1, 2), (2, 1), (4, 0)]}
ok("the Weyl formula matches the SU(3) closed form; conjugate pairs (p,q)/(q,p) always "
   "share a dimension, and for p, q <= 4 only dimension 15 is shared by unrelated irreps")

  dim   1 : [(0, 0)]
  dim   3 : [(0, 1), (1, 0)]
  dim   6 : [(0, 2), (2, 0)]
  dim   8 : [(1, 1)]
  dim  10 : [(0, 3), (3, 0)]
  dim  15 : [(0, 4), (1, 2), (2, 1), (4, 0)]
  dim  24 : [(1, 3), (3, 1)]
  dim  27 : [(2, 2)]
  dim  35 : [(1, 4), (4, 1)]

  dimensions shared by NON-conjugate irreps: {15: [(0, 4), (1, 2), (2, 1), (4, 0)]}
✓ the Weyl formula matches the SU(3) closed form; conjugate pairs (p,q)/(q,p) always share a dimension, and for p, q <= 4 only dimension 15 is shared by unrelated irreps


**Recognise.** $(p,q)$ and $(q,p)$ always share a dimension — they are conjugates, the
quark $\mathbf3=(1,0)$ and antiquark $\bar{\mathbf3}=(0,1)$ being the first example. But
$\mathbf{15}$ is worse: $(2,1)$ and $(4,0)$ are **different** irreps with the same
dimension. A dimension label for them cannot mean anything, and `resolve_rep` is built to
say so rather than guess.

In [ ]:
# ---- MOVE 4: the resolver refuses to guess. --------------------------------
assert sun.resolve_rep(3, 6) == ((2, 0), False)
assert sun.resolve_rep(3, "3bar") == ((1, 0), True)
try:
    sun.resolve_rep(3, 15)
except ValueError as e:
    print("  ValueError:", e)
else:
    raise AssertionError("dimension 15 of SU(3) should have been rejected")
ok("an ambiguous dimension raises -- pass a Dynkin tuple instead")

  ValueError: SU(3) dimension 15 is ambiguous ((4, 0), (2, 1)); pass an explicit Dynkin tuple
✓ an ambiguous dimension raises -- pass a Dynkin tuple instead


---
## 5. Inside a representation: weights and ladders

Where do the matrices for, say, the $\mathbf 6$ of SU(3) come from? From a construction
you already know: **building an angular-momentum multiplet with ladder operators**. Start
at the highest-weight state $|j,j\rangle$ and lower with $J_-$; the matrix elements are
square roots,

$$J_\pm|j,m\rangle=\sqrt{(j\mp m)(j\pm m+1)}\;|j,m\pm1\rangle.$$

A general SU($N$) irrep is built the same way, with $N-1$ commuting "$J_3$"s. The
systematic bookkeeping is the **Gelfand–Tsetlin** construction [GT50], which is what
`sun.su_n_generators` implements.

Two ideas carry the section:

- the **Cartan generators** are the ones that commute with each other — for SU(3), $T^3$
  and $T^8$. In a good basis they are diagonal, and the pair of eigenvalues on a state is
  its **weight**. The list of weights is the continuous-group counterpart of the character
  table: it names the representation without depending on how you wrote the other
  generators (§6 makes that precise).
- the **quadratic Casimir** $C_2=\sum_aT^aT^a$ commutes with every generator, so on an
  irrep it is a multiple of the identity (Schur's lemma) — one number per irrep.

> **Before running the next cells.** The $\mathbf 6$ is the symmetric product of two
> quarks. Quark weights are the three $(T^3,T^8)$ pairs of the $\mathbf3$. What are the six
> weights of the $\mathbf6$ — and what should the ladder entries look like, given the spin
> analogy?

In [ ]:
# ---- MOVE 1: set up.  The 6 of SU(3), and its raising operator. -----------
six = sun.su_n_generators(3, (2, 0))
raiser = six[0] + sp.I * six[1]              # T^1 + i T^2, like J_+
display(raiser)

⎡0  √2  0   0  0  0⎤
⎢                  ⎥
⎢0  0   √2  0  0  0⎥
⎢                  ⎥
⎢0  0   0   0  0  0⎥
⎢                  ⎥
⎢0  0   0   0  1  0⎥
⎢                  ⎥
⎢0  0   0   0  0  0⎥
⎢                  ⎥
⎣0  0   0   0  0  0⎦

In [ ]:
# ---- MOVE 2: collect the weights. --------------------------------------------
def weights(T, cartan=(2, 7), verbose=False):
    # Joint eigenvalues of the diagonal Cartan generators, one pair per state.
    H = [T[k] for k in cartan]
    assert all(h.is_diagonal() for h in H), "Cartan generators not diagonal here"
    w = sorted(tuple(sp.nsimplify(h[i, i]) for h in H) for i in range(H[0].rows))
    if verbose:
        for wt in w:
            trace(f"  (T3, T8) = {wt}")
    return w


w3, w6 = weights(T3), weights(six, verbose=True)
# the 6 is the SYMMETRIC product of two quarks: its weights are the sums w_i + w_j, i <= j
sym_sums = sorted(tuple(sp.nsimplify(x + y) for x, y in zip(w3[i], w3[j]))
                  for i in range(3) for j in range(i, 3))
assert w6 == sym_sums
ok("the six weights of the 6 are exactly the pairwise sums of quark weights, i <= j")

  (T3, T8) = (-1, sqrt(3)/3)
  (T3, T8) = (-1/2, -sqrt(3)/6)
  (T3, T8) = (0, -2*sqrt(3)/3)
  (T3, T8) = (0, sqrt(3)/3)
  (T3, T8) = (1/2, -sqrt(3)/6)
  (T3, T8) = (1, sqrt(3)/3)
✓ the six weights of the 6 are exactly the pairwise sums of quark weights, i <= j


In [ ]:
# ---- MOVE 3 + 4: sqrt ladders, closure, Schur, and the Dynkin index. -------
assert sp.sqrt(2) in raiser.values()
assert closes(six, f3)
C2_six = sp.expand(sum((Ta * Ta for Ta in six), sp.zeros(6, 6)))
assert C2_six == sp.Rational(10, 3) * sp.eye(6)
S_six = sp.nsimplify(sp.trace(six[0] * six[0]))
assert sp.Matrix(8, 8, lambda a, b: sp.trace(six[a] * six[b])) == S_six * sp.eye(8)
print(f"  C2(6) = 10/3,  Tr(T^a T^b) = {S_six} delta^ab")
ok("the ladder built sqrt(2)'s, the 6 obeys the SAME f as the 3, C2 is 10/3 times the "
   "identity (Schur), and its index is S(6) = 5/2")

  C2(6) = 10/3,  Tr(T^a T^b) = 5/2 delta^ab
✓ the ladder built sqrt(2)'s, the 6 obeys the SAME f as the 3, C2 is 10/3 times the identity (Schur), and its index is S(6) = 5/2


**Recognise.** The √2's are the spin story's square roots, now inside a colour
representation. And note what the last assertion says about §2's "judgment": the $\mathbf 6$
has $\mathrm{Tr}(T^aT^b)=\tfrac52\delta^{ab}$, not $\tfrac12$. Nobody chose $\tfrac52$ — it is
the **Dynkin index** $S(R)$, forced by using the fundamental's $f$. Normalisation is a
choice exactly once.

---
## 6. What survives a change of basis

A representation can be written in any basis: $T^a\to U\,T^aU^\dagger$ with $U$ unitary
describes the same physics — you have just relabelled the components of the multiplet.
Papers do this all the time, often silently.

> **Before running the next cell.** Under $T^a\to UT^aU^\dagger$, which of these survive:
> the individual matrix entries; the structure constants; $C_2$; the Dynkin index
> $\mathrm{Tr}(T^aT^b)$; the *set* of eigenvalues of each generator? Decide for each before
> the cell checks all of them.

In [ ]:
# ---- MOVE 1: an exact unitary -- two 3-4-5 rotations and a phase. ---------
c, s = sp.Rational(3, 5), sp.Rational(4, 5)
U = (sp.Matrix([[c, -s, 0], [s, c, 0], [0, 0, 1]]) * sp.diag(1, sp.I, 1)
     * sp.Matrix([[1, 0, 0], [0, c, -s], [0, s, c]]))
assert sp.simplify(U * U.H) == sp.eye(3)
T3_U = [sp.expand(U * Ta * U.H) for Ta in T3]
trace("U T^1 U^dagger =", T3_U[0])

U T^1 U^dagger =


⎡      -3⋅ⅈ       ⎤
⎢ 0    ─────  6/25⎥
⎢       10        ⎥
⎢                 ⎥
⎢3⋅ⅈ              ⎥
⎢───     0    8/25⎥
⎢10               ⎥
⎢                 ⎥
⎣6/25  8/25    0  ⎦

In [ ]:
# ---- MOVE 2 + 4: what changed, and what did not. ----------------------------
casimir = lambda T: sp.expand(sum((Ta * Ta for Ta in T), sp.zeros(*T[0].shape)))
spectrum = lambda M: sorted(sp.nsimplify(e) for e, k in M.eigenvals().items()
                            for _ in range(k))

assert any(A != B for A, B in zip(T3, T3_U))                      # entries: changed
assert closes(T3_U, f3)                                           # f: same
assert casimir(T3_U) == casimir(T3) == sp.Rational(4, 3) * sp.eye(3)
assert sp.Matrix(8, 8, lambda a, b: sp.trace(T3_U[a] * T3_U[b])) == sp.eye(8) / 2
assert all(spectrum(A) == spectrum(B) for A, B in zip(T3, T3_U))  # eigenvalues: same
ok("a change of basis rewrites every entry, and leaves f, C2 = 4/3, the index 1/2 and "
   "every generator's eigenvalues untouched")

✓ a change of basis rewrites every entry, and leaves f, C2 = 4/3, the index 1/2 and every generator's eigenvalues untouched


**The judgment to keep:** generator *entries* are a convention; $f$, $C_2$, the Dynkin index,
the eigenvalue spectra — and so the weights — are the representation. When two papers'
generators disagree, compare those invariants, never the matrices. It is the same lesson
`DiscreteGroups_Tutorial` §7 draws for Clebsch–Gordan coefficients versus characters. One
catch: in the rotated basis $T^3$ and $T^8$ are no longer diagonal, so reading weights off
the diagonal (as §5's helper does) is itself a basis choice — the *eigenvalues* are what
survive.

---
## 7. Conjugates, and three kinds of reality

For every representation $R$ there is a **conjugate** $\bar R$ — the one the
*antiparticles* live in. If $\psi\to e^{i\alpha^aT^a}\psi$ then
$\psi^*\to e^{-i\alpha^a(T^a)^*}\psi^*$, so the conjugate's generators are

$$\bar T^{\,a} = -\,(T^a)^* .$$

**Trap:** the minus sign. $(T^a)^*$ alone does *not* obey the algebra with the same $f$.

Whether $\bar R$ is really different from $R$ is a basis-independent question — is there
*any* $S$ with $S\,\bar T^aS^{-1}=T^a$? — and it has three answers:

| $R$ | $\bar T=T$ in some basis? | example |
|---|---|---|
| **real** | yes, and $\bar T=T$ literally in a real basis | the $\mathbf 8$ of SU(3) |
| **pseudo-real** | equivalent, $S\bar TS^{-1}=T$, but never literally equal | the $\mathbf 2$ of SU(2) |
| **complex** | no $S$ exists | the $\mathbf 3$ of SU(3) |

> **Before running the next cell.** §6 said eigenvalues survive any change of basis.
> $\bar T^8=-(T^8)^*$ has the *negated* eigenvalues of $T^8$. What does that let you conclude
> about the $\mathbf3$ versus the $\bar{\mathbf3}$ — and why can the same argument never rule
> out equivalence for SU(2)'s doublet?

In [ ]:
# ---- MOVE 1 + 2: the conjugate generators, and the sign trap. ----------------
bar = lambda T: [-Ta.conjugate() for Ta in T]
T3bar = SU3c.generators("3bar")
assert T3bar == bar(T3)
assert closes(T3bar, f3)
assert not closes([Ta.conjugate() for Ta in T3], f3)
ok("feynlag's 3bar is -T*, which closes with the shared f; T* without the sign does not")

✓ feynlag's 3bar is -T*, which closes with the shared f; T* without the sign does not


In [ ]:
# ---- MOVE 3 + 4: complex, real, pseudo-real -- decided by invariants. ------
# complex: the 3 and 3bar have different T8 spectra, so NO S can relate them
assert spectrum(T3[7]) != spectrum(T3bar[7])
print("  T8 spectrum on 3   :", spectrum(T3[7]))
print("  T8 spectrum on 3bar:", spectrum(T3bar[7]))

# real: the octet's generators are -i f, so -(T)^* = T literally
assert bar(adj_hand) == adj_hand

# pseudo-real: the doublet's conjugate differs, but eps = i sigma_2 maps it back
T2bar = bar(T2)
eps = sp.I * sigma[1]
assert T2bar != T2
assert all(eps * Tb * eps.inv() == Ta for Ta, Tb in zip(T2, T2bar))
display(sp.Eq(sp.Symbol(r"\epsilon = i\sigma_2"), eps, evaluate=False))
ok("3 is complex (spectra differ), 8 is real (-T* = T), and SU(2)'s 2 is pseudo-real: "
   "eps (-T*) eps^-1 = T")

  T8 spectrum on 3   : [-sqrt(3)/3, sqrt(3)/6, sqrt(3)/6]
  T8 spectrum on 3bar: [-sqrt(3)/6, -sqrt(3)/6, sqrt(3)/3]


                      ⎡0   1⎤
\epsilon = i\sigma₂ = ⎢     ⎥
                      ⎣-1  0⎦

✓ 3 is complex (spectra differ), 8 is real (-T* = T), and SU(2)'s 2 is pseudo-real: eps (-T*) eps^-1 = T


**Recognise.** The pseudo-real doublet is not a curiosity — it is in the Standard Model.
The Higgs $H$ is an SU(2) doublet, and $H^*$ transforms with $\bar T=-T^*$. Multiplying by
$\epsilon=i\sigma_2$ turns it back into an ordinary doublet, $\tilde H=i\sigma_2H^*$, which is
exactly how up-type quarks get a Yukawa coupling from the same Higgs. SU(3) has no such
trick: a $\bar{\mathbf 3}$ is a different particle from a $\mathbf 3$, which is why
antiquarks are not quarks.

---
## 8. Counting singlets before building them

Now the question from the introduction. A term in a Lagrangian is allowed only if it is a
**singlet**. For fields in representations $R_1,R_2,\dots$ the product transforms with

$$T^a_{R_1\otimes R_2} = T^a_{R_1}\otimes\mathbb 1 + \mathbb 1\otimes T^a_{R_2},$$

and a singlet is a vector every such generator annihilates. Two routes to the count:

- **recognise** — $C_2$ on the product splits it into eigenspaces; each eigenvalue belongs
  to an irrep (§5), and eigenspace dimension ÷ irrep dimension is how often it appears;
- **construct** — the singlets are the common null space of all the generators.

> **Before running the next cells.** Hadrons are colour singlets. Predict, for colour
> SU(3): can a quark and an antiquark ($\mathbf3\otimes\bar{\mathbf3}$) form one? Two quarks
> ($\mathbf3\otimes\mathbf3$)? Three ($\mathbf3\otimes\mathbf3\otimes\mathbf3$)? Two gluons
> ($\mathbf8\otimes\mathbf8$)? Mesons, baryons and glueballs are the answer key — and the
> absence of free diquarks is a prediction.

In [ ]:
# ---- MOVE 1: the tools -- product generators, a C2 table, the null space. --
def tensor_generators(*reps):
    # T^a on R1 x R2 x ... : one identity everywhere except the k-th slot, summed.
    dims = [R[0].shape[0] for R in reps]
    out = []
    for a in range(len(reps[0])):
        total = None
        for k in range(len(reps)):
            factors = [reps[j][a] if j == k else sp.eye(dims[j]) for j in range(len(reps))]
            term = factors[0]
            for F in factors[1:]:
                term = TensorProduct(term, F)
            total = term if total is None else total + term
        out.append(total)
    return out


def singlets(T):
    return sp.Matrix.vstack(*T).nullspace()


# C2 of the small SU(3) irreps, read off feynlag's own matrices (not typed in)
C2_TABLE = {}
for dyn in [(0, 0), (1, 0), (0, 1), (1, 1), (2, 0), (0, 2), (3, 0), (0, 3), (2, 2)]:
    Tdyn = sun.su_n_generators(3, dyn)
    C2_TABLE.setdefault(casimir(Tdyn)[0, 0], []).append((dyn, sun.weyl_dim(3, dyn)))
for val, irreps in sorted(C2_TABLE.items()):
    print(f"  C2 = {str(val):<5} : {irreps}")

  C2 = 0     : [((0, 0), 1)]
  C2 = 4/3   : [((1, 0), 3), ((0, 1), 3)]
  C2 = 3     : [((1, 1), 8)]
  C2 = 10/3  : [((2, 0), 6), ((0, 2), 6)]
  C2 = 6     : [((3, 0), 10), ((0, 3), 10)]
  C2 = 8     : [((2, 2), 27)]


In [ ]:
# ---- MOVE 2: recognise the content of a product from its C2 eigenspaces. ---
def casimir_content(T, verbose=False):
    C, n = casimir(T), T[0].shape[0]
    content, seen = {}, 0
    for val, irreps in sorted(C2_TABLE.items()):
        k = n - (C - val * sp.eye(n)).rank()          # eigenspace dimension
        if k:
            d = irreps[0][1]
            content[tuple(dyn for dyn, _ in irreps)] = k // d
            seen += k
            if verbose:
                names = " or ".join(str(dyn) for dyn, _ in irreps)
                trace(f"  C2 = {str(val):<5} eigenspace dim {k:>2} = {k // d} x dim-{d} {names}")
    assert seen == n, "an eigenvalue is missing from C2_TABLE"
    return content


T8 = adj_hand
products = {"3 x 3bar": (T3, T3bar), "3 x 3": (T3, T3),
            "3 x 3 x 3": (T3, T3, T3), "8 x 8": (T8, T8)}
counts = {}
for name, reps in products.items():
    trace(f"--- {name} ---")
    Tp = tensor_generators(*reps)
    content = casimir_content(Tp, verbose=True)
    counts[name] = (content.get(((0, 0),), 0), len(singlets(Tp)))

--- 3 x 3bar ---
  C2 = 0     eigenspace dim  1 = 1 x dim-1 (0, 0)
  C2 = 3     eigenspace dim  8 = 1 x dim-8 (1, 1)


--- 3 x 3 ---
  C2 = 4/3   eigenspace dim  3 = 1 x dim-3 (1, 0) or (0, 1)
  C2 = 10/3  eigenspace dim  6 = 1 x dim-6 (2, 0) or (0, 2)


--- 3 x 3 x 3 ---


  C2 = 0     eigenspace dim  1 = 1 x dim-1 (0, 0)


  C2 = 3     eigenspace dim 16 = 2 x dim-8 (1, 1)
  C2 = 6     eigenspace dim 10 = 1 x dim-10 (3, 0) or (0, 3)


--- 8 x 8 ---


  C2 = 0     eigenspace dim  1 = 1 x dim-1 (0, 0)


  C2 = 3     eigenspace dim 16 = 2 x dim-8 (1, 1)
  C2 = 6     eigenspace dim 20 = 2 x dim-10 (3, 0) or (0, 3)


  C2 = 8     eigenspace dim 27 = 1 x dim-27 (2, 2)


In [ ]:
# ---- MOVE 4: the two routes must agree. -------------------------------------
for name, (by_casimir, by_nullspace) in counts.items():
    print(f"  {name:<10} singlets:  from C2 = {by_casimir}   from the null space = {by_nullspace}")
    assert by_casimir == by_nullspace
assert [counts[k][1] for k in products] == [1, 0, 1, 1]
ok("mesons (3 x 3bar), baryons (3 x 3 x 3) and two-gluon states each have exactly one "
   "colour singlet; two quarks have none")

  3 x 3bar   singlets:  from C2 = 1   from the null space = 1
  3 x 3      singlets:  from C2 = 0   from the null space = 0
  3 x 3 x 3  singlets:  from C2 = 1   from the null space = 1
  8 x 8      singlets:  from C2 = 1   from the null space = 1
✓ mesons (3 x 3bar), baryons (3 x 3 x 3) and two-gluon states each have exactly one colour singlet; two quarks have none


**The baryon singlet, as an object.** A null-space vector has 27 components, one per
colour triple $(i,j,k)$. Put them in a $3\times3\times3$ array and look at which are
non-zero, and with what sign.

In [ ]:
baryon = singlets(tensor_generators(T3, T3, T3))[0]
{(i, j, k): baryon[9 * i + 3 * j + k]
 for i, j, k in itertools.product(range(3), repeat=3) if baryon[9 * i + 3 * j + k] != 0}

{(0, 1, 2): -1, (0, 2, 1): 1, (1, 0, 2): 1, (1, 2, 0): -1, (2, 0, 1): -1, (2,  ↪

↪ 1, 0): 1}

Six non-zero entries, on the six permutations of $(0,1,2)$, all equal to the sign of the
permutation times one common factor (a null-space vector's normalisation is arbitrary, so
here it is $-1$): the baryon singlet **is** the Levi-Civita symbol, $\epsilon_{ijk}\,q_iq_jq_k$ —
totally antisymmetric in colour, which is why the rest of a baryon's wavefunction must be
symmetric.

**Two traps the section exposes.**

- **$C_2$ cannot tell $R$ from $\bar R$.** In $\mathbf8\otimes\mathbf8$ one eigenspace of
  dimension 20 is "$\mathbf{10}$ *or* $\overline{\mathbf{10}}$" — the Casimir alone cannot
  say how the 20 splits, because conjugates share every $C_2$. Here reality settles it:
  the $\mathbf 8$ is real (§7), so $\mathbf8\otimes\mathbf8$ equals its own conjugate and
  must hold as many $\overline{\mathbf{10}}$s as $\mathbf{10}$s — one of each. In general
  the weights, which *do* change under conjugation, are what separate them (§11 uses
  exactly that). For singlets this never matters: the singlet is its own conjugate.
- **Distinct fields versus one field.** These counts are for *distinct* legs. For a single
  field the symmetry of the product removes some combinations — $\epsilon_{ijk}q_iq_jq_k$
  vanishes identically for three copies of one commuting field — exactly the
  $\mathbf1'$ trap of `DiscreteGroups_Tutorial` §5.1.

In [ ]:
# ---- check: the singlet is epsilon, and epsilon on ONE commuting triplet vanishes
k0 = baryon[9 * 0 + 3 * 1 + 2]
assert all(baryon[9 * i + 3 * j + k] == k0 * sp.LeviCivita(i, j, k)
           for i, j, k in itertools.product(range(3), repeat=3))
q = sp.symbols("q1:4")
assert sp.expand(sum(sp.LeviCivita(i, j, k) * q[i] * q[j] * q[k]
                     for i, j, k in itertools.product(range(3), repeat=3))) == 0
ok("the 3 x 3 x 3 singlet is proportional to eps_ijk, and eps_ijk q_i q_j q_k = 0 for a "
   "single commuting triplet -- the singlet needs distinct (or anticommuting) legs")

✓ the 3 x 3 x 3 singlet is proportional to eps_ijk, and eps_ijk q_i q_j q_k = 0 for a single commuting triplet -- the singlet needs distinct (or anticommuting) legs


---
## 9. Anomalies: a consistency test the representations must pass

Not every collection of representations gives a healthy theory. In a **chiral** gauge
theory — left- and right-handed fermions in different representations, like the real
world — the **triangle anomaly** spoils gauge invariance unless it cancels between the
fermions [GG72]. Each representation contributes a cubic **anomaly coefficient** $A(R)$,
fixed by a symmetric trace of three generators and normalised to $A(\text{fundamental})=1$:

$$\mathrm{Tr}\big(T^a_R\{T^b_R,T^c_R\}\big) = A(R)\;\mathrm{Tr}\big(T^a_F\{T^b_F,T^c_F\}\big).$$

> **Before running the next cell.** Use §7. For a **real** representation, $\bar T=T$ up to
> a change of basis; for the conjugate, $\bar T=-T^*$. What must $A(\mathbf 8)$ be, and how
> must $A(\bar{\mathbf3})$ relate to $A(\mathbf 3)$? (Traces are basis-independent, and a
> trace of three $-T^*$'s is minus the conjugate of a trace of three $T$'s.)

In [ ]:
# ---- MOVE 1 + 2: the coefficient, written out. -------------------------------
def anomaly_index(T, TF, verbose=False):
    # A(R) = Tr(T_R^a {T_R^b, T_R^c}) / Tr(T_F^a {T_F^b, T_F^c}) at the first
    # index triple where the fundamental's symmetric trace is non-zero.
    n = len(TF)
    for a, b, c in itertools.combinations_with_replacement(range(n), 3):
        tF = sp.nsimplify(sp.expand(sp.trace(TF[a] * (TF[b] * TF[c] + TF[c] * TF[b]))))
        if tF != 0:
            tR = sp.nsimplify(sp.expand(sp.trace(T[a] * (T[b] * T[c] + T[c] * T[b]))))
            if verbose:
                trace(f"  normalising on (a,b,c) = ({a + 1},{b + 1},{c + 1}):  "
                      f"Tr_R = {tR},  Tr_F = {tF}")
            return sp.simplify(tR / tF)
    return sp.S.Zero


A = {"3": anomaly_index(T3, T3, verbose=True), "3bar": anomaly_index(T3bar, T3),
     "6": anomaly_index(six, T3), "8": anomaly_index(adj_hand, T3)}
print("  ", A)

  normalising on (a,b,c) = (1,1,8):  Tr_R = sqrt(3)/6,  Tr_F = sqrt(3)/6
   {'3': 1, '3bar': -1, '6': 7, '8': 0}


In [ ]:
# ---- MOVE 3 + 4: the predictions from the box, and a basis-change check. ---
assert A == {"3": 1, "3bar": -1, "6": 7, "8": 0}
assert anomaly_index(T3_U, T3) == 1                 # the rotated 3 of section 6
ok("A(3bar) = -A(3), the real octet has A = 0, A(6) = 7 -- and a change of basis "
   "leaves A alone")

✓ A(3bar) = -A(3), the real octet has A = 0, A(6) = 7 -- and a change of basis leaves A alone


**The famous example.** The **SU(5)** grand unified theory [GG74] puts one whole generation
of left-handed matter into a $\bar{\mathbf5}$ plus a $\mathbf{10}$, the $\mathbf{10}$ being the
Dynkin label $(0,1,0,0)$. Each is anomalous on its own. Predict their sum before the cell.

In [ ]:
# ---- the SU(5) generation: two anomalous irreps, one anomaly-free theory ---
SU5 = SUN(5, "SU5")
T5 = SU5.generators(5)
A5bar = anomaly_index(SU5.generators("5bar"), T5)
A10 = anomaly_index(SU5.generators((0, 1, 0, 0)), T5)
print(f"  A(5bar) = {A5bar},  A(10) = {A10}")
assert (A5bar, A10) == (-1, 1)

five_bar = WeylFermion("fbar", reps={SU5: "5bar"}, chirality="L",
                       nflavors=1, component_names=[f"fb_{i}" for i in range(5)])
ten = WeylFermion("ften", reps={SU5: (0, 1, 0, 0)}, chirality="L",
                  nflavors=1, component_names=[f"ft_{i}" for i in range(10)])
generation = Model("SU5generation", gauge_groups=[SU5], fields=[five_bar, ten])
assert check_anomaly_free(generation).ok
ok("A(5bar) + A(10) = 0, and check_anomaly_free agrees on the actual model")

  A(5bar) = -1,  A(10) = 1
✓ A(5bar) + A(10) = 0, and check_anomaly_free agrees on the actual model


That cancellation is not an accident of bookkeeping — it is one of the reasons SU(5) was
taken seriously as a unifying group. And `check_anomaly_free` reached it on fields declared
directly in the $\bar{\mathbf5}$ and, by Dynkin tuple, the $\mathbf{10}$: a representation the
library never had hard-coded.

---
## 10. Into a model: a scalar in a brand-new representation

None of this matters unless the representations can build a Lagrangian. Give a scalar the
**fundamental of SU(4)** — a group `feynlag` never had built in — and form its covariant
derivative $D_\mu S=\partial_\mu S-ig\,A^a_\mu T^aS$.

> **Before running the next cells.** Which quadratic term in $S$ is gauge invariant, and
> how would §8 have told you without writing it? (Hint: what does $S^\dagger$ transform
> as?)

In [ ]:
# ---- MOVE 1: the covariant derivative uses the SU(4) generators -----------
g4 = ExternalParameter("g4", 0.9, positive=True)
SU4c = SUN(4, "SU4c", coupling=g4)
S = Scalar("S", reps={SU4c: 4}, component_names=[f"S_{i}" for i in range(4)])
DS = Dmu(S)
DS[0]      # first component of the covariant derivative

  ⅈ⋅SU4c₁⋅S₁⋅g₄   SU4c₁₀⋅S₃⋅g₄   √6⋅ⅈ⋅SU4c₁₅⋅S₀⋅g₄   SU4c₂⋅S₁⋅g₄   ⅈ⋅SU4c₃⋅S₀⋅ ↪
- ───────────── - ──────────── - ───────────────── - ─────────── - ─────────── ↪
        2              2                12                2              2     ↪

↪ g₄   ⅈ⋅SU4c₄⋅S₂⋅g₄   SU4c₅⋅S₂⋅g₄   √3⋅ⅈ⋅SU4c₈⋅S₀⋅g₄   ⅈ⋅SU4c₉⋅S₃⋅g₄          ↪
↪ ── - ───────────── - ─────────── - ──────────────── - ───────────── + Partia ↪
↪            2              2               6                 2                ↪

↪        
↪ lMu(S₀)
↪        

In [ ]:
# ---- MOVE 3 + 4: predict the singlet, then let the library check the term --
T4 = SU4c.generators(4)
assert len(singlets(tensor_generators(bar(T4), T4))) == 1
L = Lagrangian().add((S.dag() * S.mat)[0], sector="potential")
model = Model("SU4-scalar", gauge_groups=[SU4c], fields=[S],
              parameters=[g4], lagrangian=L)
assert model.check_invariance().ok
ok("4bar x 4 holds exactly one singlet, and S^dagger S -- that singlet -- passes "
   "feynlag's end-to-end gauge-invariance check")

✓ 4bar x 4 holds exactly one singlet, and S^dagger S -- that singlet -- passes feynlag's end-to-end gauge-invariance check


---
## 11. Capstone: the $\mathbf 6$ of SU(3) by hand

§5 took the $\mathbf 6$ from `sun.su_n_generators`. Now build it with nothing but the
$\mathbf3$ and §8's tensor product, and hold the library's construction to account.

$\mathbf3\otimes\mathbf3$ is nine-dimensional. Its **symmetric** part — spanned by $e_ie_i$
and $(e_ie_j+e_je_i)/\sqrt2$ — is six-dimensional, and the generators never mix symmetric
with antisymmetric tensors (they act the same way on both slots). So restricting
$T\otimes\mathbb1+\mathbb1\otimes T$ to the symmetric subspace gives six $6\times6$
matrices.

> **Before running the next cells.** Which tests from this notebook could tell you that
> your hand-built matrices are "the same $\mathbf6$" as the library's, even though the two
> constructions use different bases? §6 lists the candidates; §7 says which of them can
> also tell the $\mathbf 6$ from the $\bar{\mathbf6}$.

In [ ]:
# ---- MOVE 1: project the product onto its symmetric part --------------------
sym_basis = []
for i in range(3):
    for j in range(i, 3):
        v = sp.zeros(9, 1)
        if i == j:
            v[3 * i + i] = 1
        else:
            v[3 * i + j] = v[3 * j + i] = 1 / sp.sqrt(2)
        sym_basis.append(v)
Psym = sp.Matrix.hstack(*sym_basis)                       # 9 x 6, orthonormal columns
T33 = tensor_generators(T3, T3)
six_hand = [sp.simplify(Psym.H * Ta * Psym) for Ta in T33]

# the symmetric subspace really is invariant: P P^dagger commutes with every generator
assert all(sp.simplify(Ta * Psym * Psym.H - Psym * Psym.H * Ta) == sp.zeros(9, 9)
           for Ta in T33)
display(six_hand[0])

⎡    √2                 ⎤
⎢0   ──   0   0    0   0⎥
⎢    2                  ⎥
⎢                       ⎥
⎢√2           √2        ⎥
⎢──  0    0   ──   0   0⎥
⎢2            2         ⎥
⎢                       ⎥
⎢0   0    0   0   1/2  0⎥
⎢                       ⎥
⎢    √2                 ⎥
⎢0   ──   0   0    0   0⎥
⎢    2                  ⎥
⎢                       ⎥
⎢0   0   1/2  0    0   0⎥
⎢                       ⎥
⎣0   0    0   0    0   0⎦

In [ ]:
# ---- MOVE 2 + 4: every basis-independent test, against the library's 6 ------
assert closes(six_hand, f3)
assert casimir(six_hand) == sp.Rational(10, 3) * sp.eye(6)
assert sp.Matrix(8, 8, lambda a, b: sp.trace(six_hand[a] * six_hand[b])) == sp.Rational(5, 2) * sp.eye(8)
assert anomaly_index(six_hand, T3) == 7
assert weights(six_hand) == weights(six)
assert weights(six_hand) != weights(SU3c.generators("6bar"))
ok("the hand-built 6 closes with the shared f and matches the library's Gelfand-Tsetlin 6 "
   "on C2 = 10/3, S = 5/2, A = 7 and every weight -- and its weights rule out the 6bar")

✓ the hand-built 6 closes with the shared f and matches the library's Gelfand-Tsetlin 6 on C2 = 10/3, S = 5/2, A = 7 and every weight -- and its weights rule out the 6bar


In [ ]:
# ---- and the remainder: 9 = 6 + 3bar, as section 8 found --------------------
anti_basis = []
for i, j in itertools.combinations(range(3), 2):
    v = sp.zeros(9, 1)
    v[3 * i + j], v[3 * j + i] = 1 / sp.sqrt(2), -1 / sp.sqrt(2)
    anti_basis.append(v)
Panti = sp.Matrix.hstack(*anti_basis)
three_hand = [sp.simplify(Panti.H * Ta * Panti) for Ta in T33]
assert closes(three_hand, f3)
assert casimir(three_hand) == sp.Rational(4, 3) * sp.eye(3)
assert spectrum(sp.Matrix(three_hand[7])) == spectrum(T3bar[7])
assert spectrum(sp.Matrix(three_hand[7])) != spectrum(T3[7])
ok("the antisymmetric remainder is a 3bar -- two quarks make a 6 and a 3bar, never a singlet")

✓ the antisymmetric remainder is a 3bar -- two quarks make a 6 and a 3bar, never a singlet


**Recognise.** The weights did what $C_2$ could not in §8: both the $\mathbf6$ and the
$\bar{\mathbf6}$ have $C_2=10/3$, but only the $\mathbf6$ has the hand-built matrices' weights.
And the leftover antisymmetric piece is not another $\mathbf3$ but a $\bar{\mathbf3}$ — two
quarks in an antisymmetric colour state carry antiquark colour, the seed of every
diquark model.

`sun.su_n_generators` never saw a tensor product; the construction above never saw a
Gelfand–Tsetlin pattern. They agree on every invariant the notebook knows.

---
## 12. Recap

**The mechanics.**

1. A Lie group is its generators exponentiated; hermitian and traceless is what "SU" means
   to first order, and exponentiating makes it exact.
2. The structure constants $f^{abc}$ can be *derived* from any faithful set of generators.
   Every representation shares them — which fixes every normalisation after the first.
3. An irrep is named by a Dynkin label; dimensions can be ambiguous.
4. Weights, $C_2$, the Dynkin index and $A(R)$ survive any change of basis; generator
   entries do not.

**The tools, in the order you should reach for them.**

| question | tool |
|---|---|
| what are the generators of this irrep? | `SUN(N, name).generators(label)`, `sun.su_n_generators` |
| what are the structure constants? | `structure_constants`, or derive: $-2i\,\mathrm{Tr}([T^a,T^b]T^c)$ |
| which irrep is this label / dimension? | `sun.weyl_dim`, `sun.resolve_rep` |
| is this the same irrep as that one? | weights and $C_2$ — never generator entries |
| is $\bar R$ a different particle? | spectra of $\bar T=-T^*$ (complex / real / pseudo-real) |
| how many singlets in a product? | $C_2$ eigenspaces, cross-checked by the null space |
| is the fermion content consistent? | $A(R)$, `check_anomaly_free` |
| is the term I wrote invariant? | `Model.check_invariance()` |

**The traps**, each of which a check above caught:

- **Ambiguous dimensions.** $\mathbf{15}$ of SU(3) is two different irreps; pass a Dynkin
  tuple.
- **The conjugate's sign.** $\bar T=-T^*$; $T^*$ alone does not close.
- **Three kinds of reality.** A pseudo-real rep ($\mathbf2$ of SU(2)) is equivalent to its
  conjugate without being equal to it — hence $\tilde H=i\sigma_2H^*$. A complex one
  ($\mathbf3$ of SU(3)) is not equivalent at all.
- **Basis conventions.** Compare invariants across papers, never generator entries.
- **$C_2$ is blind to conjugation.** $\mathbf{10}$ and $\overline{\mathbf{10}}$ share it; weights
  tell them apart.
- **Distinct legs versus one field.** $\epsilon_{ijk}q_iq_jq_k$ is the baryon singlet for
  distinct quarks and vanishes for one commuting triplet.

**And the model-building punchline.** Before writing a term you can know whether it can
exist — a meson yes, a diquark no, a baryon exactly one way, an SU(5) generation
anomaly-free — and `feynlag` built every representation involved, including the SU(4)
fundamental and the $\mathbf{10}$ of SU(5), on demand. For more depth, Slansky's review
[Slansky81] tabulates Dynkin labels, indices and anomalies for all simple groups, and
Georgi's book [Georgi99] develops the same material for particle physicists.

Where to go next: `DiscreteGroups_Tutorial` runs the same questions for finite flavour
groups; `SM_Feynman_Rules_Tutorial` puts SU(3)×SU(2)×U(1) to work; `tests/test_sun.py` pins
the library side of everything here.

---
## 13. References

- **[Weyl25]** H. Weyl, *Theorie der Darstellung kontinuierlicher halb-einfacher Gruppen
  durch lineare Transformationen. I*, Math. Z. **23** (1925) 271–309,
  doi:[10.1007/BF01506234](https://doi.org/10.1007/BF01506234); parts II–III in Math. Z.
  **24** (1926) 328–395.
- **[GT50]** I. M. Gelfand and M. L. Tsetlin, *Finite-dimensional representations of the
  group of unimodular matrices*, Dokl. Akad. Nauk SSSR **71** (1950) 825–828 (in Russian).
- **[GG72]** H. Georgi and S. L. Glashow, *Gauge Theories Without Anomalies*, Phys. Rev. D
  **6** (1972) 429, doi:[10.1103/PhysRevD.6.429](https://doi.org/10.1103/PhysRevD.6.429).
- **[GG74]** H. Georgi and S. L. Glashow, *Unity of All Elementary-Particle Forces*, Phys.
  Rev. Lett. **32** (1974) 438,
  doi:[10.1103/PhysRevLett.32.438](https://doi.org/10.1103/PhysRevLett.32.438).
- **[Slansky81]** R. Slansky, *Group theory for unified model building*, Phys. Rept.
  **79** (1981) 1–128,
  doi:[10.1016/0370-1573(81)90092-2](https://doi.org/10.1016/0370-1573(81)90092-2).
- **[Georgi99]** H. Georgi, *Lie Algebras in Particle Physics: From Isospin to Unified
  Theories*, 2nd ed., Frontiers in Physics (Perseus Books, 1999),
  doi:[10.1201/9780429499210](https://doi.org/10.1201/9780429499210).